# 01 Data Fusion and Exploration

Build a unified payload dataset from `data/raw` by standardizing all sources to two columns: `payload` and `label`.

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path('../data/raw')

# Common column-name candidates across your raw files
PAYLOAD_CANDIDATES = ['Sentence', 'sentence', 'URL', 'url', 'payload', 'request']
LABEL_CANDIDATES = ['Label', 'label', 'classification', 'Class', 'class']

# For tabular CICIDS-style files with no payload text column:
# turn each row into a text payload from all non-label columns.
INCLUDE_TABULAR_AS_TEXT = True

In [2]:
def read_csv_flexible(path: Path) -> pd.DataFrame:
    for encoding in [None, 'utf-16', 'utf-8-sig', 'latin1']:
        try:
            if encoding is None:
                return pd.read_csv(path)
            return pd.read_csv(path, encoding=encoding)
        except Exception:
            continue
    raise ValueError(f'Could not read {path}')


def find_label_column(df: pd.DataFrame):
    for candidate in LABEL_CANDIDATES:
        if candidate in df.columns:
            return candidate
    # fallback: match after stripping spaces (for columns like ' Label')
    normalized = {str(col).strip().lower(): col for col in df.columns}
    for candidate in LABEL_CANDIDATES:
        key = candidate.strip().lower()
        if key in normalized:
            return normalized[key]
    return None


def find_payload_column(df: pd.DataFrame):
    for candidate in PAYLOAD_CANDIDATES:
        if candidate in df.columns:
            return candidate
    normalized = {str(col).strip().lower(): col for col in df.columns}
    for candidate in PAYLOAD_CANDIDATES:
        key = candidate.strip().lower()
        if key in normalized:
            return normalized[key]
    return None


def row_to_payload_text(df: pd.DataFrame, label_col: str) -> pd.Series:
    feature_cols = [c for c in df.columns if c != label_col]

    def serialize_row(row):
        parts = []
        for c in feature_cols:
            val = row[c]
            if pd.notna(val):
                parts.append(f"{str(c).strip()}={val}")
        return " | ".join(parts)

    return df.apply(serialize_row, axis=1)


def load_and_standardize(raw_dir=RAW_DIR):
    combined_list = []
    report_rows = []

    for file_path in sorted(raw_dir.glob('*.csv')):
        file_name = file_path.name

        try:
            temp_df = read_csv_flexible(file_path)
        except Exception as ex:
            report_rows.append({
                'file': file_name,
                'status': 'skipped',
                'reason': f'read_error: {type(ex).__name__}'
            })
            continue

        label_col = find_label_column(temp_df)
        if label_col is None:
            report_rows.append({
                'file': file_name,
                'status': 'skipped',
                'reason': 'no label column detected'
            })
            continue

        payload_col = find_payload_column(temp_df)

        if payload_col is not None:
            temp_out = temp_df[[payload_col, label_col]].rename(
                columns={payload_col: 'payload', label_col: 'label'}
            )
            source_mode = f'direct:{payload_col}'
        elif INCLUDE_TABULAR_AS_TEXT:
            temp_out = pd.DataFrame({
                'payload': row_to_payload_text(temp_df, label_col),
                'label': temp_df[label_col]
            })
            source_mode = 'tabular_to_text'
        else:
            report_rows.append({
                'file': file_name,
                'status': 'skipped',
                'reason': 'no payload column and tabular conversion disabled'
            })
            continue

        temp_out['source_file'] = file_name
        combined_list.append(temp_out)
        report_rows.append({
            'file': file_name,
            'status': 'included',
            'reason': source_mode
        })

    if not combined_list:
        raise ValueError('No datasets could be loaded. Check raw files/columns.')

    master = pd.concat(combined_list, ignore_index=True).drop_duplicates()
    master = master.dropna(subset=['payload', 'label']).copy()
    master['payload'] = master['payload'].astype(str).str.strip()
    master['label'] = master['label'].astype(str).str.strip()
    master = master[master['payload'] != '']

    report_df = pd.DataFrame(report_rows)
    return master.reset_index(drop=True), report_df

In [ ]:
# Master Dataset + ingestion report
master_df, ingestion_report = load_and_standardize()

display(ingestion_report)
master_df.head()

In [ ]:
print('Shape:', master_df.shape)
print('\nLabel distribution:')
display(master_df['label'].value_counts(dropna=False))

print('\nRows per source:')
display(master_df['source_file'].value_counts())

print('\nIngestion status count:')
display(ingestion_report['status'].value_counts())

In [ ]:
# Optional: save fused dataset
out_path = Path('../data/processed/master_fused_payloads.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
master_df.to_csv(out_path, index=False)
print(f'Saved: {out_path.resolve()}')